# Phase 1 — Backbone Benchmark (uqfusion)

Trains **31 YOLO variants** (full size ladder across v8/v9/v10/11/12/26), single seed, on the Pohang VIS split, then produces **Table 1**.

**How to use:** put this notebook at the **repo root** (next to `config.yaml` and the `scripts/` folder), then run the cells **top to bottom**. Cells shell out to the already-verified scripts, so nothing here re-implements the pipeline.

The big run (Step 6) is long but **resume-safe** — if the kernel dies, just re-run that cell and it continues where it stopped.

## Step 0 — One-time setup
Install the pinned environment + the `uqfusion` package. Run once per machine.

In [ ]:
!pip install -r requirements.txt -e .

## Step 1 — Point config at your data, then verify
Edit the `paths:` block in **`config.yaml`** so `data_root` points at where the Pohang dataset lives on this server (the layout that gives `<data_root>/pohang/data_vis.yaml`). Then run the cell below — every path should print **without** a `[missing]` tag.

In [ ]:
!python -m uqfusion.config

Parameters used by the rest of the notebook. `DATA` is the config alias (`vis`); `STRIDE_YAML` is the derived training yaml that Step 4 produces (path is deterministic).

In [ ]:
DATA = "vis"                                        # config alias for the Pohang VIS yaml
STRIDE_YAML = "runs/derived/data_vis_stride2.yaml"  # produced by Step 4

## Step 2 — Environment smoke check
Confirms every dependency imports, **CUDA is True**, and all 31 variants build. Stop if this isn't green.

In [ ]:
!python scripts/smoke_env.py

## Step 3 — Split audit *(optional)*
The split was already audited GREEN during data prep, and the audit is deterministic — so this is really just a **10-second upload/paths sanity check**. A FAIL here means paths or the upload are wrong, not the split. Skip if you're confident the data transferred fully.

In [ ]:
!python scripts/audit_split.py

## Step 4 — Stride-subsample the training frames
Keeps every 2nd training frame per run (10 Hz video → near-duplicate neighbours; approved A2-6). Only trims the *training* list — val/test and source files untouched. Prints the derived yaml path (matches `STRIDE_YAML`).

In [ ]:
!python scripts/make_stride_subset.py --data {DATA}

## Step 5 — Timing dry-run *(optional)*
One epoch on the smallest model to sanity-check wall-time and that `batch` fits in memory before committing to the full grid.

In [ ]:
!python scripts/run_benchmark.py --data {STRIDE_YAML} --variants yolov8n --epochs 1

## Step 6 — Full grid: 31 variants × 1 seed
The main run. **Long-running.** Resume-safe: completed `(variant, seed)` rows are skipped, so re-running this cell after a crash continues the grid. Results append to `runs/benchmark/benchmark_results.csv`.

In [ ]:
!python scripts/run_benchmark.py --data {STRIDE_YAML}

## Step 7 — FPS / latency per variant
Times each trained model's best checkpoint on the val split (fp32 + fp16 on GPU). Writes `runs/benchmark/fps.csv`.

In [ ]:
!python scripts/measure_fps.py --data {DATA}

## Step 8 — Build Table 1
Merges the grid results + FPS into `runs/benchmark/table1.md`.

In [ ]:
!python scripts/make_table1.py

## Step 9 — View Table 1

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
display(Markdown(Path("runs/benchmark/table1.md").read_text(encoding="utf-8")))

---
**Send back:** `runs/benchmark/table1.md` (and `benchmark_results.csv` / `fps.csv` if you want the raw numbers). That closes Phase 1 → winner selection.

*IR confirmation of the top-2 variants is a separate short step, run after the winner is known — not part of this notebook.*